# Part 3: Multi-Agent System

Five agents (Planner, Vision Understanding, Prompt Engineering, Image Generation, Critique) share one pydantic state and are wired together by plain Python functions (`src/orchestrator.py`) -- no agent framework. Every agent call is timed into a JSON trace. Domain-style requests use my Part 2 LoRA (`itzmnv/data266-lab2-sd15-pokemon-lora-r32`).

**Drafted, not yet run on a GPU.** The routing, safety, prompt-engineering, critique and orchestration logic in `src/` (everything except the real model calls, which are injected as callables) was stub-tested end to end on a Mac -- see `src/test_stub.py` and Section 8 below. Model loading, the real agent wrappers, and the Gradio app are written and reviewed here but will first run on a Colab T4 in the next stage.

In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

## Setup

In [ ]:
# lab rule: all required Part 3 work MUST run on one 16 GB GPU (Colab T4); fp16 throughout.
# Everything here is also designed to fit in memory at once: the vision model is loaded
# ONCE in fp16 and kept resident; each diffusion pipeline is loaded only when its route is
# used, with enable_model_cpu_offload(), and only one is ever resident at a time (see
# get_pipeline() in Section 2).
ALLOW_OTHER_GPU = False
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
is_t4 = "T4" in gpu_name
print("GPU:", gpu_name, "| T4:", is_t4, "| device:", device)
if torch.cuda.is_available() and not is_t4 and not ALLOW_OTHER_GPU:
    raise RuntimeError(f"this notebook requires a Colab T4 GPU; got {gpu_name!r}. Set ALLOW_OTHER_GPU to override.")


### Installs (pinned versions) and the `src/` package

In [ ]:
import importlib.metadata as im

# transformers pinned to a recent main-branch commit: Qwen3-VL support landed after the
# last PyPI release at the time this was written -- check huggingface.co/Qwen/Qwen3-VL-4B-Instruct
# before running and swap in a released version once one includes Qwen3-VL.
PINS = {
    "git+https://github.com/huggingface/transformers.git": None,
    "diffusers": "0.42.1", "peft": "0.21.2", "accelerate": "1.15.0",
    "torchao": "0.18.0", "pydantic": "2.7.1", "opencv-python-headless": None,
    "torchmetrics": None, "gradio": None,
}
to_install = [pkg if ver is None else f"{pkg}=={ver}" for pkg, ver in PINS.items()]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=True)
for pkg in ("transformers", "diffusers", "peft", "accelerate", "torchao", "pydantic", "cv2", "gradio"):
    try:
        print(f"{pkg}: {im.version(pkg if pkg != 'cv2' else 'opencv-python-headless')}")
    except im.PackageNotFoundError:
        print(f"{pkg}: (import only, no distribution metadata)")


In [ ]:
# The src/ package and eval/ files are copied onto the VM with `colab upload` before this
# notebook runs (same pattern as Part 2) -- e.g.
#   colab upload -s <session> part3_agents/member_manav/src /content/src
#   colab upload -s <session> eval/p3_scenarios_manav.json /content/eval/p3_scenarios_manav.json
#   colab upload -s <session> eval/p3_redteam_manav.json /content/eval/p3_redteam_manav.json
from pathlib import Path
import sys

SRC_DIR = Path("/content") if Path("/content/src").exists() else Path(".")
assert (SRC_DIR / "src" / "orchestrator.py").exists(), "src/ not found -- upload it first (see comment above)"
sys.path.insert(0, str(SRC_DIR))
print("src/ found at", SRC_DIR / "src")


## 1. Shared state

In [ ]:
from src.state import AgentState, ImageInfo, SafetyVerdict, VisionOutput, GenerationRecord, CritiqueResult

print("AgentState fields:", list(AgentState.model_fields.keys()))


## 2. Load models

In [ ]:
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, CLIPModel, CLIPProcessor

VLM_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"   # same model family as Part 1
CLIP_ID = "openai/clip-vit-large-patch14"

# The vision model is loaded ONCE, in fp16, and kept resident for every request --
# it is small enough (4B params, ~8 GB in fp16) to share a 15 GB T4 with one
# CPU-offloaded diffusion pipeline (Section 3.4's get_pipeline() never keeps more
# than one of those resident, and frees the previous one first).
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(VLM_MODEL_ID, dtype=torch.float16).to(device)
vlm_model.eval()

# CLIP for the Critique agent's similarity score -- small (~600 MB), kept resident too.
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

print("vision model and CLIP loaded")
print("peak GPU memory so far: {:.2f} GB".format(torch.cuda.max_memory_allocated() / 1e9)
      if torch.cuda.is_available() else "no GPU")


## 3. Agents

### 3.1 Planner

In [ ]:
# The Planner's rules (keywords/patterns for all five routes and for unsafe requests,
# the real-person check, and the instruction-in-image flag) live in src/planner.py and
# need no model at all -- see Section 8's stub tests. The only thing added here is the
# vision-model fallback classifier, used ONLY when none of those rules confidently match.
from src.planner import plan as planner_plan
from src.state import ROUTES


def vlm_classify_route(request: str) -> str:
    options = ", ".join(ROUTES)
    prompt = f"Classify this user request into exactly one of: {options}. Reply with only the category name.\nRequest: {request}"
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = vlm_processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                                return_dict=True, return_tensors="pt").to(vlm_model.device)
    with torch.no_grad():
        out_ids = vlm_model.generate(**inputs, max_new_tokens=10)
    text = vlm_processor.batch_decode([out_ids[0][inputs["input_ids"].shape[1]:]],
                                       skip_special_tokens=True)[0].strip().lower()
    for r in ROUTES:
        if r in text or r.replace("_", " ") in text:
            return r
    return "unsupported"


### 3.2 Vision Understanding

In [ ]:
# The quality pre-check (too small/dark), the strict-JSON prompt, and the JSON parser
# with its plain-text fallback all live in src/vision.py and need no model -- see
# Section 8. The only GPU-touching piece is vlm_call, the real model call injected
# into run_vision().
from src.vision import run_vision


def vlm_call(image_path: str, prompt: str) -> str:
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = vlm_processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                                return_dict=True, return_tensors="pt").to(vlm_model.device)
    with torch.no_grad():
        out_ids = vlm_model.generate(**inputs, max_new_tokens=300)
    trimmed = out_ids[0][inputs["input_ids"].shape[1]:]
    return vlm_processor.batch_decode([trimmed], skip_special_tokens=True,
                                       clean_up_tokenization_spaces=False)[0]


### 3.3 Prompt Engineering

In [ ]:
# Pure logic, no model -- already stub-tested in Section 8.
from src.prompt_engineering import build_generation_prompt, apply_revision

print(build_generation_prompt("Make this look like a watercolor painting.", None, "style_change"))


### 3.4 Image Generation

In [ ]:
import gc
import time
from diffusers import (StableDiffusionImg2ImgPipeline, StableDiffusionPipeline,
                        StableDiffusionInstructPix2PixPipeline)
from src.state import GenerationRecord

SD15_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
PIX2PIX_ID = "timbrooks/instruct-pix2pix"
LORA_REPO = "itzmnv/data266-lab2-sd15-pokemon-lora-r32"

_active_pipe_name, _active_pipe = None, None


def get_pipeline(name: str):
    """Loads exactly one diffusion pipeline at a time, with enable_model_cpu_offload()
    so it never competes with the resident VLM/CLIP for VRAM, and frees the previous
    pipeline first -- two diffusion pipelines are never resident together."""
    global _active_pipe_name, _active_pipe
    if _active_pipe_name == name:
        return _active_pipe
    if _active_pipe is not None:
        del _active_pipe
        gc.collect()
        torch.cuda.empty_cache()
        _active_pipe, _active_pipe_name = None, None

    if name in ("style_change", "domain_style_img2img"):
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(SD15_ID, dtype=torch.float16)
    elif name == "domain_style_text2img":
        pipe = StableDiffusionPipeline.from_pretrained(SD15_ID, dtype=torch.float16)
    elif name == "edit":
        pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(PIX2PIX_ID, dtype=torch.float16)
    else:
        raise ValueError(f"unknown pipeline: {name}")

    if name.startswith("domain_style"):
        pipe.load_lora_weights(LORA_REPO, weight_name="pytorch_lora_weights.safetensors")
    # safety checker stays on (the pipeline default) for every route
    pipe.enable_model_cpu_offload()
    _active_pipe_name, _active_pipe = name, pipe
    return pipe


GEN_DIR = Path("/content/p3_outputs/images")
GEN_DIR.mkdir(parents=True, exist_ok=True)
_gen_counter = {"n": 0}


def generate_fn(state, prompt, negative_prompt, seed, strength) -> GenerationRecord:
    t0 = time.perf_counter()
    generator = torch.Generator(device=device).manual_seed(seed)
    _gen_counter["n"] += 1
    out_path = GEN_DIR / f"gen_{_gen_counter['n']:04d}.png"

    if state.route == "style_change":
        pipe = get_pipeline("style_change")
        init_image = Image.open(state.image.path).convert("RGB").resize((512, 512))
        out = pipe(prompt=prompt, negative_prompt=negative_prompt, image=init_image,
                   strength=strength, guidance_scale=7.5, generator=generator)
        rec = GenerationRecord(model="stable-diffusion-v1-5 (img2img)", mode="img2img",
                                prompt=prompt, negative_prompt=negative_prompt, seed=seed,
                                strength=strength, guidance_scale=7.5, output_path=str(out_path))

    elif state.route == "edit":
        pipe = get_pipeline("edit")
        init_image = Image.open(state.image.path).convert("RGB").resize((512, 512))
        image_guidance_scale = 1.0 + strength  # instruct-pix2pix's analogue of "strength"
        out = pipe(prompt=prompt, image=init_image, negative_prompt=negative_prompt,
                   image_guidance_scale=image_guidance_scale, guidance_scale=7.5, generator=generator)
        rec = GenerationRecord(model="timbrooks/instruct-pix2pix", mode="instruct_edit",
                                prompt=prompt, negative_prompt=negative_prompt, seed=seed,
                                image_guidance_scale=image_guidance_scale, guidance_scale=7.5,
                                output_path=str(out_path))

    elif state.route == "domain_style":
        if state.image.path:
            pipe = get_pipeline("domain_style_img2img")
            init_image = Image.open(state.image.path).convert("RGB").resize((512, 512))
            out = pipe(prompt=prompt, negative_prompt=negative_prompt, image=init_image,
                       strength=strength, guidance_scale=7.5, generator=generator)
            mode = "img2img"
        else:
            pipe = get_pipeline("domain_style_text2img")
            out = pipe(prompt=prompt, negative_prompt=negative_prompt, height=512, width=512,
                       num_inference_steps=30, guidance_scale=7.5, generator=generator)
            mode = "text2img"
        rec = GenerationRecord(model=f"stable-diffusion-v1-5 + {LORA_REPO}", mode=mode,
                                prompt=prompt, negative_prompt=negative_prompt, seed=seed,
                                strength=strength if mode == "img2img" else None,
                                guidance_scale=7.5, output_path=str(out_path))
    else:
        raise ValueError(f"no generation pipeline for route={state.route!r}")

    image = out.images[0]
    rec.nsfw_flagged = bool(getattr(out, "nsfw_content_detected", [False])[0]) if getattr(out, "nsfw_content_detected", None) else False
    image.save(out_path)
    rec.time_s = time.perf_counter() - t0
    return rec


### 3.5 Critique

In [ ]:
# The accept/revise/reject decision logic lives in src/critique.py and needs no model
# -- see Section 8. The CLIP score and the VLM yes/no judgement are real model calls,
# defined here, plus the threshold calibration.
from src.critique import decide as critique_decide


def clip_score_fn(image_path: str, prompt: str) -> float:
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(text=[prompt], images=[image], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = clip_model(**inputs)
    img_emb = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt_emb = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((img_emb * txt_emb).sum().clamp(min=0))


def vlm_judge_fn(image_path: str, prompt: str) -> str:
    question = f'Does this image match: "{prompt}"? Answer with only yes or no.'
    raw = vlm_call(image_path, question)
    return "yes" if "yes" in raw.strip().lower() else "no"


# Calibrate the CLIP acceptance threshold from the 5 Part 2 test-split images set aside
# for exactly this (never the scenarios -- see src/build_test_images.py). This runs
# after Section 7 has built /content/p3_images.
CALIBRATION_DIR = Path("/content/p3_images")
calib_files = sorted(CALIBRATION_DIR.glob("calib_pokemon_*.png"))
if calib_files:
    calib_scores = [clip_score_fn(str(f), "a Pokemon-style creature illustration") for f in calib_files]
    CLIP_THRESHOLD = round(min(calib_scores) - 0.03, 2)  # a bit below the weakest true match
    print("calibration CLIP scores:", [round(s, 3) for s in calib_scores], "-> threshold:", CLIP_THRESHOLD)
else:
    CLIP_THRESHOLD = 0.25
    print("Section 7 has not run yet -- using a placeholder threshold:", CLIP_THRESHOLD)


## 4. Orchestration and revise loop

In [ ]:
# The orchestration itself (timing every call, the revise loop capped at 2, every
# failure turned into a clear message, a trace saved in every case) lives in
# src/orchestrator.py and needs no model -- see Section 8. This just wires in the
# real callables defined above.
import time
from typing import Optional
from src.orchestrator import run as agent_run

TRACE_DIR = "/content/p3_outputs/traces"


def handle_request(request: str, image_path: Optional[str] = None,
                    critique_enabled: bool = True, request_id: Optional[str] = None):
    request_id = request_id or f"req_{int(time.time() * 1000)}"
    return agent_run(
        request, image_path,
        vlm_call=vlm_call, generate_fn=generate_fn, classifier_fn=vlm_classify_route,
        clip_score_fn=clip_score_fn, vlm_judge_fn=vlm_judge_fn, clip_threshold=CLIP_THRESHOLD,
        critique_enabled=critique_enabled, trace_dir=TRACE_DIR, request_id=request_id,
    )


## 5. JSON traces

In [ ]:
import json as _json


def load_trace(request_id: str) -> dict:
    return _json.loads((Path(TRACE_DIR) / f"trace_{request_id}.json").read_text())


print("traces are saved to", TRACE_DIR, "-- one JSON file per request_id, written even on failure")


## 6. Failure handling

In [ ]:
# A quick, self-contained demonstration that a broken file produces a clear message
# and a saved trace, without crashing and without needing Section 7's full image set.
# (The full red-team broken-file cases, R09 and R10, are run for real in Section 8.)
_demo_path = "/content/_demo_empty.jpg"
Path(_demo_path).write_bytes(b"")
_demo_state = handle_request("edit this", _demo_path, request_id="demo_broken_file")
print(_demo_state.final_message)
print("trace saved:", Path(TRACE_DIR, "trace_demo_broken_file.json").exists())


## 7. Test images

In [ ]:
# Builds the ~25 reproducible scenario/red-team images (COCO by fixed id with their
# real license, Part 2 test-split calibration images, and the PIL red-team images) --
# see src/build_test_images.py. Images are never committed to GitHub; they go to a
# public Hugging Face dataset in a later stage.
from src.build_test_images import build_all as build_test_images

IMAGES_DIR = build_test_images(out="/content/p3_images", cache="/content/p3_coco_cache")
manifest = _json.loads((IMAGES_DIR / "manifest.json").read_text())
print(f"{len(manifest)} images/manifest entries ready at {IMAGES_DIR}")


## 8. Scenarios and red-team cases

In [ ]:
# Stub-model tests: the SAME routing/safety/prompt-engineering/critique/orchestration
# logic exercised here needs no model at all, so this cell runs identically on a
# laptop or on this GPU session -- see src/test_stub.py for every check (one scenario
# at a time, plus the 10 red-team cases, the revise-loop cap, and error handling).
from src.test_stub import run_all_tests

stub_images_dir = IMAGES_DIR if "IMAGES_DIR" in dir() else Path("/content/p3_images")
all_stub_checks_passed = run_all_tests(images_dir=str(stub_images_dir))
print("\nall stub checks passed:", all_stub_checks_passed)


## 9. With and without the critique loop

Run once the real agents have executed on a GPU: the same 15 scenarios through `handle_request(..., critique_enabled=False)` vs. the default, comparing acceptance rate and final CLIP scores. `critique_enabled` is already wired in Section 4 for exactly this comparison.

## 10. Evaluation

Filled in after the real agents have run on a GPU and the scenarios/red-team cases have executed for real (Section 8 above only proves the logic with stubs, not real model quality).

### 10.1 Primary: scenarios passed

### 10.2 Secondary: critic agreement and attack success

### 10.3 Human evaluation: pass/fail agreement

### 10.4 Efficiency: median seconds per request

### 10.5 Results table

## 11. Safety and system card

Written after the real red-team run: which attacks were blocked, the real-person and injection-handling results, and the model/license table, following the same shape as Part 1 and Part 2's safety write-ups.

## 12. Gradio app

In [ ]:
import gradio as gr


def gradio_run(image, request, critique_on):
    image_path = None
    if image is not None:
        image_path = "/content/p3_outputs/_gradio_input.png"
        image.save(image_path)
    state = handle_request(request, image_path, critique_enabled=critique_on)

    planner_text = f"route: {state.route}\nsafety: {state.safety.model_dump()}"
    vision_text = state.vision.model_dump_json(indent=2) if state.vision else "(not run -- refused or no image)"
    prompt_text = f"prompt: {state.prompt}\nnegative: {state.negative_prompt}" if state.prompt else "(not run)"
    generation_text = state.generation.model_dump_json(indent=2) if state.generation else "(not run)"
    critique_text = state.critique.model_dump_json(indent=2) if state.critique else "(not run)"
    trace_text = "\n".join(f"{e.agent}: {e.time_s:.2f}s -- {e.summary}" for e in state.trace)

    final_image = None
    if state.final_image_path and Path(state.final_image_path).exists():
        final_image = Image.open(state.final_image_path)

    return (planner_text, vision_text, prompt_text, generation_text, critique_text,
            trace_text, final_image, state.final_message or "")


with gr.Blocks(title="Part 3: Multi-Agent System") as demo:
    gr.Markdown("# Part 3: Multi-Agent System -- image upload + request, one panel per agent")
    with gr.Row():
        with gr.Column(scale=1):
            image_in = gr.Image(type="pil", label="Input image (optional)")
            request_in = gr.Textbox(label="Request", placeholder="e.g. Make this look like a watercolor painting.")
            critique_in = gr.Checkbox(value=True, label="Critique loop on")
            run_btn = gr.Button("Run", variant="primary")
            final_message_out = gr.Textbox(label="Final message", interactive=False)
            final_image_out = gr.Image(label="Final image", interactive=False)
        with gr.Column(scale=1):
            with gr.Accordion("Planner", open=True):
                planner_out = gr.Textbox(interactive=False, lines=3)
            with gr.Accordion("Vision Understanding", open=False):
                vision_out = gr.Textbox(interactive=False, lines=6)
            with gr.Accordion("Prompt Engineering", open=False):
                prompt_out = gr.Textbox(interactive=False, lines=3)
            with gr.Accordion("Image Generation", open=False):
                generation_out = gr.Textbox(interactive=False, lines=6)
            with gr.Accordion("Critique", open=False):
                critique_out = gr.Textbox(interactive=False, lines=4)
            with gr.Accordion("Trace", open=False):
                trace_out = gr.Textbox(interactive=False, lines=6)

    run_btn.click(
        gradio_run, inputs=[image_in, request_in, critique_in],
        outputs=[planner_out, vision_out, prompt_out, generation_out, critique_out,
                 trace_out, final_image_out, final_message_out],
    )

demo.launch(share=True)
